# STKI RAG Demo
Demo Retrieval Augmented Generation (RAG) untuk mata kuliah STKI.


## Cell 1: Setup
Import libraries and initialize Gemini client

In [ ]:
import os
import time
import json
import hashlib
from pathlib import Path
import fitz
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from google import genai
import re
from transformers import AutoTokenizer


# Load .env and get API key
load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise EnvironmentError(
        "GEMINI_API_KEY tidak ditemukan. "
        "Isi file .env dengan GEMINI_API_KEY=key-mu."
    )

client = genai.Client(api_key=GEMINI_API_KEY)

d:\Kuliah\SMT 4\STKI\STKI-RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [40]:
print(fitz.__doc__)

PyMuPDF 1.28.0: Python bindings for the MuPDF 1.29.0 library.
Python 3.11 running on linux (64-bit).



## Cell 2: Helper Functions
Dua fungsi utama yang dipakai berulang kali di notebook ini:
- `get_embedding` — ubah teks menjadi vektor (dengan retry + fallback lokal)
- `cosine_sim` — hitung cosine similarity antar dua vektor

In [3]:
EMBED_MODEL = "gemini-embedding-001"
EMBED_CACHE_PATH = Path(".cache") / "gemini_embedding_cache.json"


def load_embedding_cache() -> dict[str, list[float]]:
    if EMBED_CACHE_PATH.exists():
        return json.loads(EMBED_CACHE_PATH.read_text(encoding="utf-8"))
    return {}


def save_embedding_cache(cache: dict[str, list[float]]) -> None:
    EMBED_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    EMBED_CACHE_PATH.write_text(
        json.dumps(cache, ensure_ascii=True),
        encoding="utf-8",
    )


def _cache_key(text: str, model: str = EMBED_MODEL) -> str:
    normalized = re.sub(r"\s+", " ", text).strip()
    return hashlib.sha256(f"{model}::{normalized}".encode("utf-8")).hexdigest()


def _embed_batch_with_retry(
    texts: list[str],
    model: str = EMBED_MODEL,
    retries: int = 4,
    base_delay: float = 1.0,
    ) -> list[list[float]]:
    for attempt in range(retries):
        try:
            resp = client.models.embed_content(
                model=model,
                contents=texts,
            )
            return [item.values for item in resp.embeddings]
        except Exception as exc:
            if attempt < retries - 1:
                time.sleep(base_delay * (2 ** attempt))
            else:
                raise RuntimeError(
                    f"Gagal membuat embedding batch setelah {retries} percobaan."
                ) from exc


embedding_cache = load_embedding_cache()


def get_embeddings_batch(
    texts: list[str],
    batch_size: int = 32,
    model: str = EMBED_MODEL,
    retries: int = 4,
    base_delay: float = 1.0,
    ) -> list[list[float]]:
    """Embed list of texts with cache + batching to reduce API calls."""
    if not texts:
        return []

    results: list[list[float] | None] = [None] * len(texts)
    misses: list[tuple[int, str, str]] = []

    for idx, text in enumerate(texts):
        key = _cache_key(text, model=model)
        cached = embedding_cache.get(key)
        if cached is not None:
            results[idx] = cached
        else:
            misses.append((idx, key, text))

    for start in range(0, len(misses), batch_size):
        batch_items = misses[start : start + batch_size]
        batch_texts = [item[2] for item in batch_items]
        batch_embeddings = _embed_batch_with_retry(
            batch_texts,
            model=model,
            retries=retries,
            base_delay=base_delay,
        )

        for (idx, key, _), emb in zip(batch_items, batch_embeddings):
            embedding_cache[key] = emb
            results[idx] = emb

    if misses:
        save_embedding_cache(embedding_cache)

    return [emb for emb in results if emb is not None]


def get_embedding(text: str, retries: int = 4, base_delay: float = 1.0) -> list[float]:
    """Generate single embedding using same cached batch pipeline."""
    return get_embeddings_batch(
        [text],
        batch_size=1,
        model=EMBED_MODEL,
        retries=retries,
        base_delay=base_delay,
    )[0]


def cosine_sim(a, b) -> float:
    """Cosine similarity between two vectors.

    Returns 0.0 if either vector is zero-length to avoid division by zero.
    """
    a, b = np.array(a), np.array(b)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)


def clean_text(text: str) -> str:
    # Hilangkan spasi di akhir/belakang
    text = text.strip()

    # Hilangkan spasi sebelum newline
    text = re.sub(r"[ \t]+\n", "\n", text)

    # Maksimal dua newline (paragraf)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text


def load_pdf_text(pdf_path: str):
    doc = fitz.open(pdf_path)
    text = ""

    for page in doc:
        text += page.get_text()

    return clean_text(text)


def load_md_text(file_path: Path) -> str:
    """
    Membaca file Markdown dan mengembalikan seluruh isinya sebagai string.
    """
    text = file_path.read_text(encoding="utf-8")
    return clean_text(text)


# pdf_text = load_pdf_text("dokumen/Buku Panduan Penulisan TA Teknologi Informasi Revisi 9.pdf")
# print(pdf_text[:200])

## Cell 3: Load Query and Documents
Tentukan query (pertanyaan) dan muat dokumen dari folder `dokumen/`.

In [4]:
query = "What time do I leave for school?"


In [5]:
BASE_DIR = Path.cwd()
DOCS_DIR = BASE_DIR / "dokumen"
print(f"BASE_DIR: {BASE_DIR}")
print(f"DOCS_DIR: {DOCS_DIR}")

BASE_DIR: d:\Kuliah\SMT 4\STKI\STKI-RAG
DOCS_DIR: d:\Kuliah\SMT 4\STKI\STKI-RAG\dokumen


In [6]:
documents_name = []
for file in DOCS_DIR.iterdir():
    if file.is_file():
        documents_name.append(file.name)
        
print(documents_name)

['Artificial-Inteligence.md', 'Climate-Change.md', 'Cyber-Security.md']


In [7]:


def load_documents() -> dict[str, str]:
    """Read all files from dokumen/ directory."""
    docs = {}
    for fname in documents_name:
        file_path = DOCS_DIR / fname
        if fname.endswith(".pdf"):
            docs[fname] = load_pdf_text(file_path)
        elif fname.endswith(".md"):
            docs[fname] = load_md_text(file_path)
        else:
            print(f"Unsupported file type: {fname}")
    return docs

DOCS = load_documents()

print("Query:", query)
print("Documents loaded:", list(DOCS.keys()))
for name, text in DOCS.items():
    print(f"--- {name} ---")
    print(text[:1000])
    print()

Query: What time do I leave for school?
Documents loaded: ['Artificial-Inteligence.md', 'Climate-Change.md', 'Cyber-Security.md']
--- Artificial-Inteligence.md ---
# Artificial Intelligence and Modern Technology

## Introduction

Artificial Intelligence (AI) has become one of the most influential
technologies of the twenty-first century. It enables computers to
perform tasks that traditionally required human intelligence, including
recognizing images, understanding natural language, making predictions,
and solving complex problems. AI is now used in education, healthcare,
transportation, finance, manufacturing, entertainment, and scientific
research.

Although AI appears to be a recent innovation, its foundations were
established decades ago through advances in mathematics, computer
science, statistics, and cognitive science. The rapid increase in
computing power, cloud infrastructure, and the availability of large
datasets has accelerated AI development dramatically.

## History of Ar

In [46]:
display(DOCS)

{'Cyber-Security.md': '# Cybersecurity\n\n## Introduction\n\nCybersecurity is the practice of protecting computer systems, networks,\nsoftware, and digital information from unauthorized access, attacks, and\ndamage. As businesses and governments become increasingly dependent on\ndigital technologies, cybersecurity has become one of the most critical\nareas of information technology.\n\nA strong cybersecurity strategy combines technology, organizational\npolicies, and user awareness to reduce security risks.\n\n## Common Cyber Threats\n\nCybercriminals use various attack techniques, including phishing,\nmalware, ransomware, social engineering, denial-of-service attacks,\npassword attacks, and insider threats. These attacks may result in\nfinancial losses, data breaches, or service disruptions.\n\nAttackers continuously develop new techniques, making cybersecurity an\nongoing challenge.\n\n## Security Principles\n\nThree fundamental principles of cybersecurity are confidentiality,\ninteg

# Chunk

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
tokenizer.model_max_length = int(1e9)

d:\Kuliah\SMT 4\STKI\STKI-RAG\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Juana\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [9]:
def chunk_text(
    text: str,
    tokenizer,
    chunk_size: int = 256,
    overlap_ratio: float = 0.2,
) -> list[str]:
    """
    Melakukan chunking berdasarkan token menggunakan Sliding Window.

    Parameters
    ----------
    text : str
        Dokumen yang akan di-chunk.
    tokenizer :
        HuggingFace tokenizer.
    chunk_size : int
        Maksimal jumlah token tiap chunk.
    overlap_ratio : float
        Persentase overlap (0.0 - 1.0).

    Returns
    -------
    list[str]
        Daftar chunk dalam bentuk teks.
    """

    # Tokenisasi
    token_ids = tokenizer.encode(
    text,
    add_special_tokens=False,
    truncation=False
)

    overlap_tokens = int(chunk_size * overlap_ratio)

    step = chunk_size - overlap_tokens

    chunks = []

    for start in range(0, len(token_ids), step):

        end = start + chunk_size

        chunk_ids = token_ids[start:end]

        if not chunk_ids:
            break

        chunk_text = tokenizer.decode(
            chunk_ids,
            skip_special_tokens=True
        )

        chunks.append(chunk_text)

    return chunks

In [ ]:
# ex_text = DOCS["Lapkeu BCA Mar 26.pdf"]
# chunks = chunk_text(ex_text, tokenizer, chunk_size=256, overlap_ratio=0.2)
# print(f"Jumlah chunk : {len(chunks)}")
# # display(chunks)

KeyError: 'Lapkeu BCA Mar 26.pdf'

In [10]:
CHUNK_SIZE = 384
OVERLAP_RATIO = 0.1
BATCH_SIZE = 32

embedded_chunks = {}
total_chunks = 0

for doc_name, text in DOCS.items():
    chunks = chunk_text(
        text,
        tokenizer,
        chunk_size=CHUNK_SIZE,
        overlap_ratio=OVERLAP_RATIO,
    )

    # Deduplicate chunks per document so repeated text is embedded once.
    unique_chunks = list(dict.fromkeys(chunks))
    unique_embeddings = get_embeddings_batch(unique_chunks, batch_size=BATCH_SIZE)
    unique_map = dict(zip(unique_chunks, unique_embeddings))
    chunk_embeddings = [unique_map[c] for c in chunks]

    embedded_chunks[doc_name] = {
        "chunks": chunks,
        "embeddings": chunk_embeddings,
    }

    total_chunks += len(chunks)
    print(
        f"{doc_name}: {len(chunks)} chunks "
        f"({len(unique_chunks)} unique) embedded"
    )


print(f"Total chunks indexed: {total_chunks}")
print(f"Documents indexed: {len(embedded_chunks)}")
print(f"Cache file: {EMBED_CACHE_PATH}")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (770 > 512). Running this sequence through the model will result in indexing errors


RuntimeError: Gagal membuat embedding batch setelah 4 percobaan.

## Cell 4: Vector Search


Lakukan pencarian berbasis chunk, lalu ambil Top-K chunk paling relevan ke query.

In [ ]:
TOP_K = 5

q_vec = get_embedding(query)
print(f"Query embedded: {len(q_vec)} dimensions")

chunk_rows = []
for doc_name, payload in embedded_chunks.items():
    chunks = payload["chunks"]
    embeddings = payload["embeddings"]
    for idx, (chunk_text_value, chunk_vec) in enumerate(zip(chunks, embeddings)):
        score = cosine_sim(q_vec, chunk_vec)
        chunk_rows.append(
            {
                "document": doc_name,
                "chunk_id": idx,
                "score": score,
                "chunk_text": chunk_text_value,
            }
        )

if not chunk_rows:
    raise RuntimeError("Index chunk kosong. Jalankan Cell chunk embedding terlebih dahulu.")

chunk_scores_df = pd.DataFrame(chunk_rows).sort_values(
    by="score",
    ascending=False,
    ignore_index=True,
)

top_chunks_df = chunk_scores_df.head(TOP_K).copy()
top_chunks = top_chunks_df.to_dict(orient="records")

doc_scores = (
    chunk_scores_df.groupby("document", as_index=False)["score"]
    .max()
    .sort_values(by="score", ascending=False, ignore_index=True)
    .rename(columns={"score": "best_chunk_score"})
)

best = top_chunks[0]["document"]
best_chunk = top_chunks[0]["chunk_text"]

print("Top documents by best chunk score:")
display(doc_scores)

print(f"Top {TOP_K} chunks:")
display(top_chunks_df[["document", "chunk_id", "score"]])

print("Best chunk preview:")
print(best_chunk[:500])

Skip dokumen Cyber-Security.md karena error: Gagal membuat embedding setelah 4 percobaan.


KeyboardInterrupt: 

## Cell 5: RAG Answer Generation
Gunakan Top-K chunk hasil retrieval sebagai konteks, lalu minta Gemini menjawab secara terstruktur: ANSWER / SNIPPET / REASON.

In [ ]:
METRIC = "cosine similarity on chunk embeddings"
TOP_K_CONTEXT = 3

selected_chunks = top_chunks[:TOP_K_CONTEXT]
context_blocks = []
for rank, item in enumerate(selected_chunks, start=1):
    context_blocks.append(
        f"[Chunk {rank}] doc={item['document']} chunk_id={item['chunk_id']} score={item['score']:.4f}\n"
        f"{item['chunk_text']}"
    )
context_text = "\n\n".join(context_blocks)

prompt = (
    "You are a semantic Question Answering system. "
    "Answer using ONLY the provided chunk context. "
    "If context is insufficient, say so clearly.\n\n"
    f"Query: {query}\n"
    f"Retrieval metric: {METRIC}\n"
    f"Chunk config: chunk_size={CHUNK_SIZE}, overlap_ratio={OVERLAP_RATIO}\n\n"
    f"Context:\n{context_text}\n\n"
    "Format output exactly as:\n"
    "ANSWER: ...\n"
    "SNIPPET: ...\n"
    "REASON: ..."
    )

print("Generating answer from top chunks...")
start_answer = time.time()
resp = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=prompt,
    )
elapsed_answer = time.time() - start_answer

# Parse structured output
lines = resp.text.strip().splitlines()
rag_result = {"answer": "", "snippet": "", "reason": ""}
current = ""
for line in lines:
    if line.startswith("ANSWER:"):
        current = "answer"
        rag_result["answer"] = line.replace("ANSWER:", "").strip()
    elif line.startswith("SNIPPET:"):
        current = "snippet"
        rag_result["snippet"] = line.replace("SNIPPET:", "").strip()
    elif line.startswith("REASON:"):
        current = "reason"
        rag_result["reason"] = line.replace("REASON:", "").strip()
    elif current == "answer" and line.strip():
        rag_result["answer"] += " " + line.strip()
    elif current == "snippet" and line.strip():
        rag_result["snippet"] += " " + line.strip()
    elif current == "reason" and line.strip():
        rag_result["reason"] += " " + line.strip()

print("ANSWER:", rag_result["answer"])
print("SNIPPET:", rag_result["snippet"])
print("REASON:", rag_result["reason"])
print(f"[Generated in {elapsed_answer:.2f}s]")

Generating answer from D1 ...
ANSWER: You leave for school at around 6:30 PM.
SNIPPET: I always leave home at around 6:30 PM.
REASON: D1 is the best match because it contains the specific time mentioned in the text for leaving home, which directly addresses the user's query about their departure time.
[Generated in 1.14s]


## Cell 6: Chunking Hyperparameter Sweep
Bandingkan beberapa kombinasi `chunk_size` dan `overlap_ratio` berdasarkan skor retrieval chunk untuk query yang sama.

In [ ]:
GRID_CHUNK_SIZES = [256, 384, 512]
GRID_OVERLAPS = [0.05, 0.1, 0.2]
TOP_K_EVAL = 5

def build_chunk_index(docs: dict[str, str], chunk_size: int, overlap_ratio: float):
    raw_rows = []
    unique_chunks = []
    seen_chunks = set()

    for doc_name, text in docs.items():
        chunks = chunk_text(
            text,
            tokenizer,
            chunk_size=chunk_size,
            overlap_ratio=overlap_ratio,
        )

        for idx, chunk_text_value in enumerate(chunks):
            raw_rows.append((doc_name, idx, chunk_text_value))
            if chunk_text_value not in seen_chunks:
                seen_chunks.add(chunk_text_value)
                unique_chunks.append(chunk_text_value)

    unique_embeddings = get_embeddings_batch(unique_chunks, batch_size=BATCH_SIZE)
    emb_map = dict(zip(unique_chunks, unique_embeddings))

    chunk_index = []
    for doc_name, idx, chunk_text_value in raw_rows:
        chunk_index.append(
            {
                "document": doc_name,
                "chunk_id": idx,
                "chunk_text": chunk_text_value,
                "embedding": emb_map[chunk_text_value],
            }
        )

    return chunk_index, len(raw_rows), len(unique_chunks)


def evaluate_chunking_grid(
    query_text: str,
    docs: dict[str, str],
    chunk_sizes: list[int],
    overlaps: list[float],
    top_k: int = 5,
    ):
    q_vec = get_embedding(query_text)
    eval_rows = []

    for chunk_size in chunk_sizes:
        for overlap in overlaps:
            index_rows, total_chunks, unique_chunks = build_chunk_index(
                docs,
                chunk_size=chunk_size,
                overlap_ratio=overlap,
            )

            scored = []
            for item in index_rows:
                scored.append(
                    {
                        "document": item["document"],
                        "chunk_id": item["chunk_id"],
                        "score": cosine_sim(q_vec, item["embedding"]),
                        "chunk_text": item["chunk_text"],
                    }
                )

            scored = sorted(scored, key=lambda x: x["score"], reverse=True)
            top_items = scored[:top_k]

            top1_score = top_items[0]["score"] if top_items else 0.0
            mean_topk = (
                sum(item["score"] for item in top_items) / len(top_items)
                if top_items
                else 0.0
            )
            est_requests = (unique_chunks + BATCH_SIZE - 1) // BATCH_SIZE

            eval_rows.append(
                {
                    "chunk_size": chunk_size,
                    "overlap_ratio": overlap,
                    "total_chunks": total_chunks,
                    "unique_chunks": unique_chunks,
                    "est_embed_requests": est_requests,
                    "top1_score": top1_score,
                    "mean_topk_score": mean_topk,
                    "top_document": top_items[0]["document"] if top_items else None,
                    "top_chunk_preview": (
                        top_items[0]["chunk_text"][:160].replace("\n", " ")
                        if top_items
                        else ""
                    ),
                }
            )

    result_df = pd.DataFrame(eval_rows).sort_values(
        by=["top1_score", "mean_topk_score"],
        ascending=False,
        ignore_index=True,
    )
    return result_df


grid_result_df = evaluate_chunking_grid(
    query_text=query,
    docs=DOCS,
    chunk_sizes=GRID_CHUNK_SIZES,
    overlaps=GRID_OVERLAPS,
    top_k=TOP_K_EVAL,
    )

display(
    grid_result_df[
        [
            "chunk_size",
            "overlap_ratio",
            "total_chunks",
            "unique_chunks",
            "est_embed_requests",
            "top1_score",
            "mean_topk_score",
            "top_document",
        ]
    ]
)

print("Best config:")
display(grid_result_df.head(1))